# 03 — Generación de pares contexto-target (Fase 3)

Prueba interactiva de `src/dataset.py`. Se valida:

1. Que la **ventana esté bien construida**: pares traducidos de
   vuelta a palabras, no a índices.
2. Qué pasa en los **bordes** de la oración, donde el contexto queda corto.
3. El efecto de `context_size`, `drop_unknown` y el subsampling.
4. El **relleno y la máscara** que necesita el modelo para promediar.
5. Los dos envoltorios de PyTorch: `CBOWDataset` (materializado) y
   `CBOWIterableDataset` (streaming), y que ambos den lo mismo.

In [1]:
import sys
import time
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import torch
from torch.utils.data import DataLoader

from src.config import corpus_path, load_config, vocab_path
from src.dataset import (
    CBOWDataset,
    CBOWIterableDataset,
    format_pair,
    generate_pairs,
    pad_context,
)
from src.vocabulary import Vocabulary

CONFIG = load_config("configs/base.yaml")
CORPUS = corpus_path(CONFIG)
VOCAB = Vocabulary.load(vocab_path(CONFIG))
TOK = CONFIG["tokenizer"]
C = CONFIG["context_size"]

print("config       :", CONFIG["name"])
print("context_size :", C)
print("drop_unknown :", CONFIG["drop_unknown"])
print("subsampling  :", CONFIG["subsampling_threshold"])
print("vocabulario  :", VOCAB)
print("torch        :", torch.__version__, "| CUDA:", torch.cuda.is_available())

config       : base_5k_50_2
context_size : 2
drop_unknown : True
subsampling  : 1e-05
vocabulario  : Vocabulary(size=5000, total_tokens=2,028,209,650, coverage=0.8326, min_count=5)
torch        : 2.6.0+cu124 | CUDA: True


## 1. La mecánica de la ventana, sobre una oración de ejemplo

Para ver el mecanismo sin ruido, escribimos una oración conocida a un `.txt` y
la pasamos por la **misma** función que se usa con el corpus real
(`open_corpus` acepta `.bz2`, `.gz` o texto plano). Nada se reimplementa acá.

In [2]:
JUGUETE = ROOT / "data" / "processed" / "_demo" / "juguete.txt"
JUGUETE.parent.mkdir(parents=True, exist_ok=True)
JUGUETE.write_text("el gato juega en la casa\n", encoding="utf-8")

# Verificamos que todas las palabras estén en el vocabulario, para que el
# ejemplo no se distorsione por palabras fuera de vocabulario.
for palabra in "el gato juega en la casa".split():
    print(f"  {palabra:<8} {'en el vocabulario' if palabra in VOCAB else 'FUERA (OOV)'}")

  el       en el vocabulario
  gato     en el vocabulario
  juega    en el vocabulario
  en       en el vocabulario
  la       en el vocabulario
  casa     en el vocabulario


In [3]:
print("Oración: el gato juega en la casa")
print(f"context_size = {C}  (o sea {C} palabras a cada lado)\n")
print(f"{'contexto':<34} {'target'}")
print("-" * 46)

for context, target in generate_pairs(JUGUETE, VOCAB, C, subsampling_threshold=0, **TOK):
    palabras = " ".join(VOCAB.word(i) for i in context)
    print(f"{'[' + palabras + ']':<34} -> {VOCAB.word(target)}")

Oración: el gato juega en la casa
context_size = 2  (o sea 2 palabras a cada lado)

contexto                           target
----------------------------------------------
[gato juega]                       -> el
[el juega en]                      -> gato
[el gato en la]                    -> juega
[gato juega la casa]               -> en
[juega en casa]                    -> la
[en la]                            -> casa


Leyendo la salida de arriba se ve el mecanismo completo:

| Target | Contexto | Por qué |
|---|---|---|
| `el` | `gato juega` | es el primero: **no tiene nada a la izquierda** |
| `gato` | `el juega en` | tiene 1 a la izquierda y 2 a la derecha |
| `juega` | `el gato en la` | está en el medio: ventana **completa**, 2+2 |
| `casa` | `en la` | es el último: no tiene nada a la derecha |

Los targets del medio tienen contexto completo (4 palabras) y los de los bordes
tienen menos. **No se descartan**: el contexto se entrega con su largo real y el
relleno se resuelve después, en la capa de tensores.

> Esta decisión importa cada vez más al escalar. Con `context_size=10` y
> oraciones de 21 tokens en promedio, exigir ventana completa dejaría afuera la
> mayoría de los targets del corpus.

## 2. Diez pares reales del corpus

Lo que pide la especificación: pares generados sobre una muestra del corpus,
traducidos de vuelta a palabras para verificar visualmente la ventana.

Acá **sí** aparecen los efectos del corpus real: `drop_unknown=True` quita las
palabras fuera de vocabulario, así que puede haber saltos respecto del texto
original.

In [4]:
pares = generate_pairs(
    CORPUS, VOCAB, C,
    limit=20,
    seed=CONFIG["seed"],
    drop_unknown=CONFIG["drop_unknown"],
    subsampling_threshold=0,     # sin subsampling, para ver la ventana limpia
    **TOK,
)

for n, (context, target) in enumerate(pares, start=1):
    print(f"{n:2d}. {format_pair(context, target, VOCAB)}")
    if n >= 10:
        break

 1. [el nombre]  ->  en
 2. [en nombre de]  ->  el
 3. [en el de el]  ->  nombre
 4. [el nombre el el]  ->  de
 5. [nombre de el]  ->  el
 6. [de el]  ->  el
 7. [señor del]  ->  sea
 8. [sea del universo]  ->  señor
 9. [sea señor universo]  ->  del
10. [señor del]  ->  universo


In [5]:
# La oración original de la que salen esos pares, para comparar
from src.corpus import stream_sentences, tokenize

primera = next(stream_sentences(CORPUS, limit=1))
tokens = tokenize(primera, **TOK)
print("oración cruda :", primera)
print("tokenizada    :", tokens)
print("con drop_unknown=True:")
print("               ", VOCAB.decode(VOCAB.encode(tokens, drop_unknown=True)))

oración cruda : En el nombre de Alá el Compasivo el Misericordioso
tokenizada    : ['en', 'el', 'nombre', 'de', 'alá', 'el', 'compasivo', 'el', 'misericordioso']
con drop_unknown=True:
                ['en', 'el', 'nombre', 'de', 'el', 'el']


## 3. Efecto de `context_size`

Las opciones del proyecto contempla contextos de 2, 5 y 10. Sobre la misma oración
de ejemplo se ve cuánto se ensancha la ventana y cuántos pares salen.

In [6]:
JUGUETE2 = JUGUETE.parent / "juguete2.txt"
JUGUETE2.write_text(
    "el gobierno anunció una nueva inversión en la ciudad de buenos aires\n",
    encoding="utf-8",
)

for c in (2, 5, 10):
    pares = list(generate_pairs(JUGUETE2, VOCAB, c, subsampling_threshold=0, **TOK))
    anchos = [len(ctx) for ctx, _ in pares]
    print(f"context_size={c:2d}: {len(pares):2d} pares | "
          f"contexto de {min(anchos)} a {max(anchos)} palabras "
          f"(máximo posible {2 * c}) | promedio {sum(anchos) / len(anchos):.1f}")

context_size= 2: 12 pares | contexto de 2 a 4 palabras (máximo posible 4) | promedio 3.5
context_size= 5: 12 pares | contexto de 5 a 10 palabras (máximo posible 10) | promedio 7.5
context_size=10: 12 pares | contexto de 10 a 11 palabras (máximo posible 20) | promedio 10.8


In [7]:
# El mismo target, visto con las tres ventanas
OBJETIVO = "inversión"
for c in (2, 5, 10):
    for context, target in generate_pairs(JUGUETE2, VOCAB, c, subsampling_threshold=0, **TOK):
        if VOCAB.word(target) == OBJETIVO:
            print(f"context_size={c:2d} -> {format_pair(context, target, VOCAB)}")
            break

context_size= 2 -> [una nueva en la]  ->  inversión
context_size= 5 -> [el gobierno anunció una nueva en la ciudad de buenos]  ->  inversión
context_size=10 -> [el gobierno anunció una nueva en la ciudad de buenos aires]  ->  inversión


Se ve el compromiso: una ventana chica captura relaciones **sintácticas**
(el vecino inmediato), y una grande captura relaciones **temáticas** (de qué
habla la oración). Por eso la grilla del proyecto la varía.

## 4. `drop_unknown`: qué hacer con las palabras fuera de vocabulario

Con el vocabulario de 5.000 palabras, el **16,7% del corpus es `<UNK>`**. Hay
dos posturas y la config elige una:

* `drop_unknown=True` (lo que usamos) — el OOV se **elimina** de la oración. El
  modelo nunca gasta capacidad prediciendo un comodín sin significado, pero dos
  palabras separadas por un OOV quedan artificialmente adyacentes.
* `drop_unknown=False` — el OOV se convierte en `<UNK>` y ocupa su posición.
  Preserva las distancias, pero mete un token vacío de contenido en las ventanas.

In [8]:
JUGUETE3 = JUGUETE.parent / "juguete3.txt"
JUGUETE3.write_text("el gato zzzqwx juega en la casa\n", encoding="utf-8")
print("Oración: el gato zzzqwx juega en la casa   ('zzzqwx' no está en el vocabulario)\n")

for drop in (True, False):
    print(f"drop_unknown={drop}:")
    for context, target in generate_pairs(
        JUGUETE3, VOCAB, C, drop_unknown=drop, subsampling_threshold=0, **TOK
    ):
        print(f"   {format_pair(context, target, VOCAB)}")
    print()

Oración: el gato zzzqwx juega en la casa   ('zzzqwx' no está en el vocabulario)

drop_unknown=True:
   [gato juega]  ->  el
   [el juega en]  ->  gato
   [el gato en la]  ->  juega
   [gato juega la casa]  ->  en
   [juega en casa]  ->  la
   [en la]  ->  casa

drop_unknown=False:
   [gato <UNK>]  ->  el
   [el <UNK> juega]  ->  gato
   [el gato juega en]  ->  <UNK>
   [gato <UNK> en la]  ->  juega
   [<UNK> juega la casa]  ->  en
   [juega en casa]  ->  la
   [en la]  ->  casa



Con `drop_unknown=True`, `gato` y `juega` quedan pegados como si `zzzqwx` no
hubiera existido. Con `False`, aparecen pares cuyo target es `<UNK>` —
entrenamiento gastado en predecir un comodín.

## 5. Subsampling aplicado a la generación

El subsampling de la Fase 2 se aplica **acá**: antes de deslizar la ventana, se
descartan al azar ocurrencias de las palabras muy frecuentes. Eso no solo
achica el dataset, también **cambia las ventanas**: al quitar `de` y `la`, las
palabras con contenido quedan más cerca entre sí.

In [9]:
N_ORACIONES = 20_000

for t in (0.0, 1e-5):
    t0 = time.time()
    total = sum(
        1 for _ in generate_pairs(
            CORPUS, VOCAB, C, limit=N_ORACIONES, seed=CONFIG["seed"],
            drop_unknown=CONFIG["drop_unknown"], subsampling_threshold=t, **TOK
        )
    )
    etiqueta = "sin subsampling" if t == 0 else f"subsampling t={t:g}"
    print(f"{etiqueta:<22}: {total:>9,} pares  ({time.time() - t0:.1f}s)")

sin subsampling       :   358,066 pares  (0.7s)


subsampling t=1e-05   :    50,981 pares  (0.7s)


In [10]:
# La misma oración, con y sin subsampling
UNA = ROOT / "data" / "processed" / "_demo" / "una.txt"
UNA.write_text(
    "el ministerio de economía de la nación anunció la creación de un fondo\n",
    encoding="utf-8",
)

for t in (0.0, 1e-5):
    pares = list(generate_pairs(UNA, VOCAB, C, subsampling_threshold=t, seed=7, **TOK))
    etiqueta = "sin subsampling" if t == 0 else "con subsampling"
    print(f"{etiqueta} ({len(pares)} pares):")
    for context, target in pares[:5]:
        print(f"   {format_pair(context, target, VOCAB)}")
    print()

sin subsampling (13 pares):
   [ministerio de]  ->  el
   [el de economía]  ->  ministerio
   [el ministerio economía de]  ->  de
   [ministerio de de la]  ->  economía
   [de economía la nación]  ->  de

con subsampling (4 pares):
   [economía nación]  ->  ministerio
   [ministerio nación anunció]  ->  economía
   [ministerio economía anunció]  ->  nación
   [economía nación]  ->  anunció



## 6. Cuántos pares produce el corpus

Dato clave para dimensionar el entrenamiento de la Fase 5.

In [11]:
N_MEDICION = 50_000

t0 = time.time()
n_pares = sum(
    1 for _ in generate_pairs(
        CORPUS, VOCAB, C, limit=N_MEDICION, seed=CONFIG["seed"],
        drop_unknown=CONFIG["drop_unknown"],
        subsampling_threshold=CONFIG["subsampling_threshold"], **TOK
    )
)
elapsed = time.time() - t0

ORACIONES_CORPUS = 95_800_000
por_oracion = n_pares / N_MEDICION

print(f"oraciones medidas      : {N_MEDICION:,}")
print(f"pares generados        : {n_pares:,}")
print(f"pares por oración      : {por_oracion:.2f}")
print(f"velocidad              : {n_pares / elapsed:,.0f} pares/s")
print()
print(f"Extrapolando a las {ORACIONES_CORPUS:,} oraciones del corpus:")
print(f"  pares por época      : ~{por_oracion * ORACIONES_CORPUS:,.0f}")
print(f"  batches de {CONFIG['batch_size']}       : ~{por_oracion * ORACIONES_CORPUS / CONFIG['batch_size']:,.0f}")
print(f"  solo generarlos      : ~{ORACIONES_CORPUS / (N_MEDICION / elapsed) / 60:,.0f} min por época")

oraciones medidas      : 50,000
pares generados        : 146,557
pares por oración      : 2.93
velocidad              : 87,377 pares/s

Extrapolando a las 95,800,000 oraciones del corpus:
  pares por época      : ~280,803,212
  batches de 512       : ~548,444
  solo generarlos      : ~54 min por época


## 7. Relleno y máscara

El modelo promedia los embeddings del contexto. Como los contextos tienen largo
variable, se rellenan hasta `2 * context_size` y se acompaña una **máscara** que
vale 1 en las posiciones reales y 0 en el relleno. Así el promedio se calcula
solo sobre lo real, y el valor con el que se rellenó es irrelevante.

In [12]:
ejemplos = [
    ([12, 45], "borde: solo 2 palabras de contexto"),
    ([12, 45, 7, 9], "medio: ventana completa"),
]
for contexto, descripcion in ejemplos:
    indices, mascara = pad_context(contexto, C)
    print(f"{descripcion}")
    print(f"   contexto original : {contexto}")
    print(f"   relleno a {2 * C}       : {indices}")
    print(f"   máscara           : {mascara}")
    print(f"   promedia sobre    : {sum(mascara)} posiciones\n")

borde: solo 2 palabras de contexto
   contexto original : [12, 45]
   relleno a 4       : [12, 45, 0, 0]
   máscara           : [1, 1, 0, 0]
   promedia sobre    : 2 posiciones

medio: ventana completa
   contexto original : [12, 45, 7, 9]
   relleno a 4       : [12, 45, 7, 9]
   máscara           : [1, 1, 1, 1]
   promedia sobre    : 4 posiciones



## 8. `CBOWDataset` — pares materializados

Guarda los pares en arrays de numpy. Tiene `len()` y permite barajado real, pero
solo entra en memoria una cantidad acotada: es para notebooks, pruebas y
configuraciones chicas.

In [13]:
t0 = time.time()
dataset = CBOWDataset.from_config(CONFIG, VOCAB, max_pairs=200_000, limit=100_000)
print(dataset, f"| construido en {time.time() - t0:.1f}s")
print()

contexto, mascara, target = dataset[0]
print("un elemento del dataset (arrays de numpy, no tensores):")
print("   contexto :", contexto, contexto.dtype, tuple(contexto.shape))
print("   máscara  :", mascara, mascara.dtype)
print("   target   :", target, type(target).__name__)
print()
print("El dataset entrega numpy y el DataLoader los apila en tensores de a un")
print("batch entero: construir tensores de a un elemento cuesta la mitad del")
print("rendimiento del pipeline (medido: 93k -> 46k pares/s).")
print()
print("   en palabras:", format_pair(
    [i for i, m in zip(contexto.tolist(), mascara.tolist()) if m], int(target), VOCAB
))
print()
bytes_por_par = dataset.nbytes / len(dataset)
pares_corpus = por_oracion * ORACIONES_CORPUS      # medido en la seccion 6
print(f"memoria por par : {bytes_por_par:.1f} bytes")
print(f"pares del corpus: ~{pares_corpus:,.0f}")
print(f"si se materializaran todos: {bytes_por_par * pares_corpus / 1024**3:,.1f} GB")
print("-> por eso el corpus completo se recorre con CBOWIterableDataset")

CBOWDataset(pairs=200,000, context_size=2, memoria=4.6 MB) | construido en 2.7s

un elemento del dataset (arrays de numpy, no tensores):
   contexto : [2910    0    0    0] int64 (4,)
   máscara  : [1. 0. 0. 0.] float32
   target   : 218 int

El dataset entrega numpy y el DataLoader los apila en tensores de a un
batch entero: construir tensores de a un elemento cuesta la mitad del
rendimiento del pipeline (medido: 93k -> 46k pares/s).

   en palabras: [universo]  ->  señor

memoria por par : 24.0 bytes
pares del corpus: ~280,803,212
si se materializaran todos: 6.3 GB
-> por eso el corpus completo se recorre con CBOWIterableDataset


In [14]:
loader = DataLoader(dataset, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=0)
contextos, mascaras, targets = next(iter(loader))

print("un batch del DataLoader:")
print(f"   contextos : {tuple(contextos.shape)}  {contextos.dtype}")
print(f"   máscaras  : {tuple(mascaras.shape)}  {mascaras.dtype}")
print(f"   targets   : {tuple(targets.shape)}  {targets.dtype}")
print()
print(f"esperado  : (batch_size={CONFIG['batch_size']}, 2*context_size={2 * C})")
print(f"contextos con ventana completa en este batch: "
      f"{int((mascaras.sum(dim=1) == 2 * C).sum())} de {len(targets)}")

un batch del DataLoader:
   contextos : (512, 4)  torch.int64
   máscaras  : (512, 4)  torch.float32
   targets   : (512,)  torch.int64

esperado  : (batch_size=512, 2*context_size=4)
contextos con ventana completa en este batch: 141 de 512


## 9. `CBOWIterableDataset` — streaming

El que se usa para el corpus completo: no acumula nada, va emitiendo pares a
medida que lee. No tiene `len()` ni admite `shuffle=True`.

In [15]:
iterable = CBOWIterableDataset.from_config(CONFIG, VOCAB, limit=5_000)
print(iterable)

loader_stream = DataLoader(iterable, batch_size=CONFIG["batch_size"], num_workers=0)
contextos, mascaras, targets = next(iter(loader_stream))
print(f"\nun batch por streaming:")
print(f"   contextos : {tuple(contextos.shape)}")
print(f"   máscaras  : {tuple(mascaras.shape)}")
print(f"   targets   : {tuple(targets.shape)}")

CBOWIterableDataset(context_size=2, subsampling=1e-05, drop_unknown=True)

un batch por streaming:
   contextos : (512, 4)
   máscaras  : (512, 4)
   targets   : (512,)


In [16]:
# Los dos datasets tienen que producir exactamente los mismos pares
mismos_pares = list(generate_pairs(
    CORPUS, VOCAB, C, limit=200, seed=CONFIG["seed"],
    drop_unknown=CONFIG["drop_unknown"],
    subsampling_threshold=CONFIG["subsampling_threshold"], **TOK
))
materializado = CBOWDataset(mismos_pares, C)
streaming = CBOWIterableDataset.from_config(CONFIG, VOCAB, limit=200)

import numpy as np

iguales = all(
    np.array_equal(a[0], b[0]) and np.array_equal(a[1], b[1]) and a[2] == b[2]
    for a, b in zip(materializado, streaming)
)
print(f"pares materializados : {len(materializado):,}")
print(f"pares por streaming  : {sum(1 for _ in streaming):,}")
print(f"¿idénticos?          : {iguales}")

pares materializados : 231
pares por streaming  : 231
¿idénticos?          : True


In [17]:
# set_epoch cambia el sorteo del subsampling: cada época ve pares distintos
iterable = CBOWIterableDataset.from_config(CONFIG, VOCAB, limit=500)

for epoca in range(3):
    iterable.set_epoch(epoca)
    total = sum(1 for _ in iterable)
    primeros = [int(t) for _, _, t in list(iterable)[:5]]
    print(f"época {epoca}: {total:,} pares | primeros targets: {primeros}")

print("\nSin set_epoch, el subsampling descartaría siempre las mismas")
print("ocurrencias y todas las épocas verían exactamente los mismos pares.")

época 0: 596 pares | primeros targets: [218, 2910, 3897, 115, 965]


época 1: 585 pares | primeros targets: [10, 2910, 1512, 188, 457]
época 2: 632 pares | primeros targets: [156, 218, 2910, 324, 324]

Sin set_epoch, el subsampling descartaría siempre las mismas
ocurrencias y todas las épocas verían exactamente los mismos pares.


In [18]:
# Limpieza de los archivos de demostración
import shutil
shutil.rmtree(JUGUETE.parent)
print("archivos de demostración eliminados")

archivos de demostración eliminados


## 10. El reparto entre workers

`CBOWIterableDataset` reparte el corpus **por oración** cuando el `DataLoader`
usa varios procesos: cada worker tokeniza solo las oraciones que le tocan. La
alternativa —repartir por par— obligaría a todos a tokenizar todo y descartar lo
ajeno, o sea paralelismo con cero ganancia.

In [19]:
from src.dataset import generate_pairs as _gp

# Sin subsampling la partición tiene que ser EXACTA (con subsampling cada shard
# sortea por su cuenta, así que los conteos difieren un poco)
opciones = dict(seed=CONFIG["seed"], drop_unknown=True, subsampling_threshold=0, **TOK)
completo = list(_gp(CORPUS, VOCAB, C, limit=3000, **opciones))
partes = [
    list(_gp(CORPUS, VOCAB, C, limit=3000, shard_index=i, shard_count=3, **opciones))
    for i in range(3)
]

print(f"sin repartir  : {len(completo):,} pares")
print(f"shards 0,1,2  : {[len(p) for p in partes]} -> suma {sum(len(p) for p in partes):,}")
print(f"partición exacta      : {len(completo) == sum(len(p) for p in partes)}")
print(f"los mismos pares      : "
      f"{sorted(map(str, completo)) == sorted(map(str, [x for s in partes for x in s]))}")

sin repartir  : 32,385 pares
shards 0,1,2  : [10498, 10959, 10928] -> suma 32,385
partición exacta      : True
los mismos pares      : True


> **Medido en esta máquina, conviene igual `num_workers=0`.** Con 2, 4 y 6
> workers el rendimiento *baja* (43k → 37k → 31k → 25k pares/s): mandar los
> tensores entre procesos cuesta más que tokenizar. El reparto por oración deja
> la opción disponible y correcta, pero no es la que gana acá.

## Conclusiones de la Fase 3

- La ventana se construye correctamente: verificado palabra por palabra sobre
  una oración conocida y sobre pares reales del corpus.
- Los targets de los **bordes** se conservan con contexto corto en vez de
  descartarse; el relleno y la máscara resuelven el largo variable en la capa de
  tensores. Sin esto, `context_size=10` perdería la mayoría del corpus.
- `drop_unknown=True` evita entrenar contra un comodín sin significado.
- El subsampling se aplica antes de deslizar la ventana, así que además de
  achicar el dataset acerca entre sí a las palabras con contenido.
- `CBOWDataset` y `CBOWIterableDataset` producen exactamente los mismos pares;
  el primero es para pruebas, el segundo para el corpus completo.
- Ambos entregan arrays de numpy y dejan que el `DataLoader` arme los tensores
  por batch, lo que casi duplica el rendimiento del pipeline.